# Week 4: Predictive Modeling and Optimization in Logistics Systems

Uses existing `data/cleaned_logistics_data.csv`; no additional dataset was created. Provenance caveat: see `data/DATASET_SOURCE.md`. Target: `time_taken_min` (minutes).

## Problem, features, and data preparation
Predict duration from approximate distance, vehicle condition, multiple-delivery count, order hour/day, driver age, traffic, weather, vehicle type, city, festival, and weekday. Exclude IDs, raw coordinates, ratings, pickup times and pickup-derived variables, and anything derived from the target. Numeric missing values use training medians; missing categories map to `Unknown`; encoding and imputation fit only within training folds. Feature availability at dispatch is an assumption to verify.

In [ ]:
from pathlib import Path
import json, pandas as pd
ROOT=Path.cwd()
if not (ROOT/'data').exists(): ROOT=ROOT.parent
results=json.loads((ROOT/'reports/week4_results.json').read_text())
data=pd.read_csv(ROOT/results['dataset']['path'], low_memory=False)
print(data.shape)

Loaded **45,584** rows over 44 distinct dates, 2022-02-11 through 2022-04-06. Chronological split: training before 2022-03-29, 36,138 training and 9,446 test rows (seed 42).

In [ ]:
# Run the complete reproducible workflow. This writes the CV, test metrics,
# predictions, triage summary, and charts used in the following sections.
import sys
sys.path.insert(0, str(ROOT/'src'))
from week4_predictive_modeling import run
results, cv_folds, test_comparison, cv_summary, predictions, actual = run()

## Models and validation
Compare linear regression, a depth-limited decision tree, and two random-forest settings. Three expanding-window validation folds on training dates select the candidate with lowest mean RMSE. Tree fallback tuning caps each CV training sample at 5,000 and validation at 3,500; final tree fitting uses a fixed-seed sample of 15,000, while linear regression uses all training rows. The later-date test set is held out from selection.

In [ ]:
cv=pd.read_csv(ROOT/'reports/week4_cross_validation.csv')
cv.groupby('model')[['MAE','RMSE','R2']].mean().sort_values('RMSE')

CV summary, computed by the analysis run:

| Model | Mean CV RMSE | Mean CV MAE | Mean CV R? |
| Decision Tree (depth 8) | 7.150 | 5.661 | 0.426 |
| Linear Regression | 6.452 | 5.131 | 0.533 |
| Random Forest (12 trees) | 6.053 | 4.825 | 0.589 |
| Random Forest (20 trees) | 5.705 | 4.542 | 0.635 |

## Test performance
MAE and RMSE are minutes; R? is unitless. Model selection used CV RMSE, not test metrics. Test results below are from the executed script.

In [ ]:
pd.read_csv(ROOT/'reports/week4_model_comparison.csv')

| Model | MAE min | RMSE min | R? |
|---|---:|---:|---:|
| Linear Regression | 5.171 | 6.501 | 0.531 |
| Decision Tree (depth 8) | 5.852 | 7.435 | 0.386 |
| Random Forest (12 trees) | 4.582 | 5.790 | 0.628 |
| Random Forest (20 trees) | 4.182 | 5.294 | 0.689 |

![Test metrics](../visualizations/week4_model_comparison.png)

Figure 1. Chronological test metrics.

![Actual versus predicted](../visualizations/week4_actual_vs_predicted.png)

Figure 2. Selected model predictions against observed test durations.

## Capacity-constrained triage demonstration
Objective: maximize long-duration cases surfaced with review capacity fixed at 20%. The threshold is the 32-minute training-period 75th percentile. The model ranks the test set by predicted duration; the baseline selects the same 1,889 orders randomly with a fixed seed. All selected records are reviewed at most once. Model queue surfaced 1,589/2,640 long cases (recall 60.2%; precision 84.1%); random queue surfaced 549 (recall 20.8%; precision 29.1%). This ranks review opportunities only; it does not change durations or quantify time saved.

![Triage](../visualizations/week4_triage_optimization.png)

## Interpretation, limitations, and reproduction
The dataset is a short historical 2022 window, and exact source identity/license remain unverified. Straight-line distance is not road distance. Inputs such as multiple-delivery count and vehicle condition need decision-time verification. Scikit-learn and Jupyter execution tools were unavailable: the script used the existing NumPy linear/tree fallback; compact fallback forests are educational implementations. Validation used sampled tree subsets. This single historical holdout does not guarantee current accuracy. Triage is a retrospective capacity simulation, not real operational optimization. Reproduce with `python src/week4_predictive_modeling.py` from project root. Results are under `reports/week4_*`; charts under `visualizations/week4_*`.

## Execution note
Jupyter execution tools are not installed in this environment, so this notebook has valid JSON and runnable cells but no cached cell outputs. The equivalent analysis script was executed twice, and its metrics, CSV/JSON artifacts, and charts were validated. The DOCX is valid OOXML with embedded charts; `python-docx` is also unavailable here, so it could not be reopened through that library.